In [1]:
import open3d as o3d
import rospy
import rosbag
import numpy as np

pcl_rosbag_path = '/home/seungwoo/alchemist_ws/src/BPMP-Tracker/script/airsim_results/airsim_pcl_logging2.bag'
bag = rosbag.Bag(pcl_rosbag_path)

pcl_list = []
for topic, msg, t in bag.read_messages(topics=['/airsim/lopcl']):
        if msg._type == 'sensor_msgs/PointCloud2':
            pcl_msg = msg
            pcl = o3d.geometry.PointCloud()
            pcl.points = o3d.utility.Vector3dVector(np.frombuffer(pcl_msg.data, dtype=np.float32).reshape(-1, 3))
            pcl_list.append(pcl)

pcl_rosbag_path = '/home/seungwoo/alchemist_ws/src/BPMP-Tracker/script/airsim_results/airsim_pcl_logging.bag'
bag = rosbag.Bag(pcl_rosbag_path)

for topic, msg, t in bag.read_messages(topics=['/airsim/lopcl']):
        if msg._type == 'sensor_msgs/PointCloud2':
            pcl_msg = msg
            pcl = o3d.geometry.PointCloud()
            pcl.points = o3d.utility.Vector3dVector(np.frombuffer(pcl_msg.data, dtype=np.float32).reshape(-1, 3))
            pcl_list.append(pcl)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


INFO - 2025-10-13 19:29:22,278 - topics - topicmanager initialized


In [2]:
o3d.visualization.draw_geometries(pcl_list)

In [3]:
pcl_all = o3d.geometry.PointCloud()
for i, pcl in enumerate(pcl_list):
    pcl_all += pcl

# pcl_all = pcl_all.voxel_down_sample(voxel_size=0.05)
pcl_all = pcl_all.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)[0]


# z = 0 으로 투영
pcl_array = np.asarray(pcl_all.points)
pcl_array[:,2] = 0.0
pcl_all.points = o3d.utility.Vector3dVector(pcl_array)
pcl_all = pcl_all.voxel_down_sample(voxel_size=0.05)


o3d.visualization.draw_geometries([pcl_all])


In [4]:

# ===============================
# 1. 데이터 로드 (사용자 준비)
# ===============================
# 예시: rgb_list, depth_list, vo_poses는 미리 준비된 리스트
#  - rgb_list: (H, W, 3) numpy array
#  - depth_list: (H, W) numpy array, 사람 영역은 mask된 depth
#  - vo_poses: 각 프레임의 VO pose (4x4 numpy array)

import cv2
from tqdm import tqdm
import rosbag
import numpy as np
from tf.transformations import euler_from_quaternion

def pose2d_to_mat4(x, y, yaw):
    """
    2D pose (x, y, yaw) → 4x4 homogeneous transform (SE3)
    yaw 단위: rad
    """
    c, s = np.cos(yaw), np.sin(yaw)
    T = np.eye(4)
    T[0,0], T[0,1] = c, -s
    T[1,0], T[1,1] = s,  c
    T[0,3], T[1,3] = x, y
    return T
vo_poses = []      # VO trajectory (각각 4x4 pose)
vo_time = []  
target_pose = []  # 각 프레임의 객체 검출 결과 (3D 좌표)
target_pose_time = []
dynamic_obj = [ [] for i in range(10)]  # 각 프레임의 객체 검출 결과 (사용자 정의 형식)
dynamic_obj_time = [ [] for i in range(10)]


bagfile = "/home/seungwoo/alchemist_ws/src/BPMP-Tracker/script/airsim_results/first_success.bag"
odom_topic = "/airsim/odom"
target_pose_topic = "/airsim/object/target/pose"
dynamic_obj_topic = ["/airsim/object/dynamic_{}/pose".format(i+1) for i in range(10)]


with rosbag.Bag(bagfile) as bag:
    for topic, msg, t in tqdm(bag.read_messages(topics=[odom_topic, target_pose_topic] + dynamic_obj_topic), desc="Loading data from bag",):
        if topic == odom_topic:    
            x = msg.pose.pose.position.x
            y = msg.pose.pose.position.y
            z = 0  # 2D이므로 z=0 고정
            # orientation quaternion -> yaw
            q = msg.pose.pose.orientation
            quat = [q.x, q.y, q.z, q.w]
            roll, pitch, yaw = euler_from_quaternion(quat)
            T = pose2d_to_mat4(x, y, yaw)
            vo_poses.append(T)
            vo_time.append(t)
        elif topic == target_pose_topic:
            object_detect_xyz = np.array([msg.pose.position.x, msg.pose.position.y, msg.pose.position.z], dtype=np.float32)
            target_pose.append(object_detect_xyz)
            target_pose_time.append(t)
        else:
            for i in range(10):
                if topic == dynamic_obj_topic[i]:
                    obj_xyz = np.array([msg.pose.position.x, msg.pose.position.y, msg.pose.position.z], dtype=np.float32)
                    dynamic_obj[i].append(obj_xyz)
                    dynamic_obj_time[i].append(t)
                    break

# 시간 동기화 (가장 가까운 시간의 depth/rgb/vo 매칭)
synced_vo_poses = []
synced_target_pose = []
synced_dynamic_obj = [ [] for i in range(10)]
for i in range(len(vo_time)):
    vo_t = vo_time[i]
    for j in range(10):
        obj_diffs = [abs((vo_t - ot).to_sec()) for ot in dynamic_obj_time[j]]
        obj_idx = np.argmin(obj_diffs)
        synced_dynamic_obj[j].append(dynamic_obj[j][obj_idx])

    target_diffs = [abs((vo_t - tt).to_sec()) for tt in target_pose_time]
    target_idx = np.argmin(target_diffs)
    vo_idx = i
    synced_vo_poses.append(vo_poses[vo_idx])
    synced_target_pose.append(target_pose[target_idx])


vo_poses = np.array(synced_vo_poses)
target_pose = np.array(synced_target_pose)
dynamic_obj = np.array(synced_dynamic_obj)


Loading data from bag: 29189it [00:01, 23918.12it/s]


In [ ]:
# import pickle
# with open('/home/seungwoo/alchemist_ws/src/BPMP-Tracker/script/airsim_results/vo_target_dynamic.pkl', 'wb') as f:
#     pickle.dump({
#         'vo_poses': vo_poses,
#         'target_pose': target_pose,
#         'dynamic_obj': dynamic_obj,
#     }, f)

# with open('/home/seungwoo/alchemist_ws/src/BPMP-Tracker/script/airsim_results/vo_target_dynamic.pkl', 'rb') as f:
#     data = pickle.load(f)
# vo_poses = data['vo_poses']
# target_pose = data['target_pose']
# dynamic_obj = data['dynamic_obj']   

In [5]:

import matplotlib.pyplot as plt
%matplotlib qt
start = 0
end = len(target_pose)

angle = np.deg2rad(-90)

rotation_matrix = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])   
target_pose_rot = target_pose.copy()
target_pose_rot[:, :2] = target_pose[:, :2] @ rotation_matrix.T
vo_poses_rot = vo_poses.copy()
vo_poses_rot[:, :2, 3] = vo_poses[:, :2, 3] @ rotation_matrix.T
dynamic_obj_rot = dynamic_obj.copy()
for i in range(dynamic_obj.shape[0]):
    dynamic_obj_rot[i][:, :2] = dynamic_obj[i][:, :2] @ rotation_matrix.T
pcl_array_rot = pcl_array.copy()
pcl_array_rot[:, :2] = pcl_array[:, :2] @ rotation_matrix.T
plt.figure(figsize=(5,3))
plt.axis('equal')
plt.plot(target_pose_rot[start:end,0], target_pose_rot[start:end,1], 'r-', label='Object Detections')
plt.plot(vo_poses_rot[start:end,0, 3], vo_poses_rot[start:end,1, 3], 'b-', label='ICP refined (accumulated)')


In [6]:

for i in range(dynamic_obj.shape[0]):
    # 지날수록 점점더 연하게
    for j in range(len(dynamic_obj_rot[i][start:end,0])):
        alpha = (j+1) / len(dynamic_obj_rot[i][start:end,0])
        plt.plot(dynamic_obj_rot[i][j:j+1,0], dynamic_obj_rot[i][j:j+1,1], color=(0,1,0,alpha))


plt.scatter(pcl_array_rot[:,0], pcl_array_rot[:,1], s=5, c=(0.5,0.5,0.5), label='Point Cloud')

# plt.title("Trajectory and Object Detections")
# plt.xlabel("X (m)")
# plt.ylabel("Y (m)")

plt.xlim([-35, 2.5])
plt.ylim([-3, 17])


plt.grid()
# plt.legend()


: 